# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baselsalah342-max/flyrank_intern/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*



**One row (raw grain)** = one content item, for one client, on one calendar day
(`report_date` × `client_hash_id` × `content_hash_id`), from `fact_content_daily_performance`.

**Tables used:**
- `fact_content_daily_performance` (month=2026-03 partition) — daily GSC/GA4 signals
- `dim_content` — static content attributes (word_count, content_created_date, etc.), joined
  on `client_hash_id` + `content_hash_id`

**Time window:** March 2026 (`month=2026-03`), split into two halves around a decision point:
- Feature window: 2026-03-01 → 2026-03-15 (what we know)
- Label window: 2026-03-16 → 2026-03-31 (what happens next)

This is a mid-panel month (per the internship data-use rule), never the sealed `_sample`
final month.

In [25]:

import os
import duckdb
from dotenv import load_dotenv
import pandas as pd
load_dotenv()  # reads .env from the current working directory (repo root)


HF_TOKEN = os.environ.get("HF_TOKEN")
assert HF_TOKEN, "HF_TOKEN not found — check your .env file exists and has HF_TOKEN=..."

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"

# 1. see the real column names + types for the daily fact table
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet') LIMIT 0")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [26]:

pd.set_option("display.max_rows", None)

BASE = "hf://datasets/FlyRank/internship-warehouse"

print("=== fact_content_daily_performance ===")
fact_cols = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet') LIMIT 0").df()
print(fact_cols[["column_name", "column_type"]].to_string())

print("\n=== dim_content ===")
content_cols = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{BASE}/dim_content.parquet') LIMIT 0").df()
print(content_cols[["column_name", "column_type"]].to_string())

=== fact_content_daily_performance ===
                 column_name column_type
0                report_date        DATE
1             client_hash_id     VARCHAR
2            content_hash_id     VARCHAR
3             client_has_gsc     BOOLEAN
4             client_has_ga4     BOOLEAN
5         gsc_data_available     BOOLEAN
6         ga4_data_available     BOOLEAN
7            gsc_impressions      BIGINT
8                 gsc_clicks      BIGINT
9           gsc_sum_position      BIGINT
10          gsc_avg_position      DOUBLE
11             ga4_pageviews      BIGINT
12              ga4_sessions      BIGINT
13                 ga4_users      BIGINT
14      ga4_engaged_sessions      BIGINT
15  ga4_total_engagement_sec      BIGINT
16          sessions_organic      BIGINT
17           sessions_direct      BIGINT
18         sessions_referral      BIGINT
19           sessions_social      BIGINT
20             sessions_paid      BIGINT
21               sessions_ai      BIGINT
22                

In [27]:
print(fact_cols[["column_name", "column_type"]].to_string())

                 column_name column_type
0                report_date        DATE
1             client_hash_id     VARCHAR
2            content_hash_id     VARCHAR
3             client_has_gsc     BOOLEAN
4             client_has_ga4     BOOLEAN
5         gsc_data_available     BOOLEAN
6         ga4_data_available     BOOLEAN
7            gsc_impressions      BIGINT
8                 gsc_clicks      BIGINT
9           gsc_sum_position      BIGINT
10          gsc_avg_position      DOUBLE
11             ga4_pageviews      BIGINT
12              ga4_sessions      BIGINT
13                 ga4_users      BIGINT
14      ga4_engaged_sessions      BIGINT
15  ga4_total_engagement_sec      BIGINT
16          sessions_organic      BIGINT
17           sessions_direct      BIGINT
18         sessions_referral      BIGINT
19           sessions_social      BIGINT
20             sessions_paid      BIGINT
21               sessions_ai      BIGINT
22                ai_chatgpt      BIGINT
23             a

In [28]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{BASE}/dim_content.parquet') LIMIT 0")

┌────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name         │ column_type │  null   │   key   │ default │  extra  │
│          varchar           │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ url_hash_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_char_count         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_token_count        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ url_char_count             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ content_created_date       │ DATE        │ YES     │ NULL    │ 

In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


print("fact_content_daily_performance columns:", fact_cols["column_name"].tolist())
print("\ndim_content columns:", content_cols["column_name"].tolist())

fact_content_daily_performance columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']

dim_content columns: ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


**Context (join/group only, never features):**
- `client_hash_id`, `content_hash_id` — pseudonymized ids

**Features (knowable at the decision moment, 2026-03-15):**
- `gsc_impressions`, `gsc_avg_position`, `ga4_engaged_sessions` — from the feature window only
  (2026-03-01 to 2026-03-15)
- `word_count`, `content_created_date` — static `dim_content` attributes

**Label / proxy (what I predict):**
- `is_declining` — a proxy I define myself: 1 if total GSC impressions in the label window
  (03-16→03-31) fall more than 20% below the feature-window total, else 0. This mirrors the
  starter dataset's `trend_direction == "down"` logic (ML-03), now built on real daily warehouse
  data instead of a pre-computed CSV column.

**Excluded (deliberately, with why):**
- All AI-referral columns (`sessions_ai`, `ai_chatgpt`, `ai_perplexity`, `ai_gemini`,
  `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other`) — across the whole warehouse only ~30K of
  78.8M rows have any AI-referral signal, so for a single mid-panel month these columns are
  overwhelmingly zero and would just be noise for this lane. They belong to the separate
  "AI Referral Opportunity" lane, not Refresh/Content Opportunity Scoring.

In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show why the excluded AI-referral columns are the right call to exclude: check their density
# in our actual slice (month=2026-03), not just the whole-warehouse number from the guide.
ai_density_sql = f"""
SELECT
  COUNT(*) AS total_rows,
  COUNT(*) FILTER (WHERE sessions_ai > 0) AS rows_with_ai_sessions,
  ROUND(100.0 * COUNT(*) FILTER (WHERE sessions_ai > 0) / COUNT(*), 4) AS pct_with_ai_sessions
FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
"""
con.sql(ai_density_sql)

┌────────────┬───────────────────────┬──────────────────────┐
│ total_rows │ rows_with_ai_sessions │ pct_with_ai_sessions │
│   int64    │         int64         │        double        │
├────────────┼───────────────────────┼──────────────────────┤
│    9841378 │                  5534 │               0.0562 │
└────────────┴───────────────────────┴──────────────────────┘

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

 Verify it with queries (grain, counts, availability) + 5 features + the leakage trap

In [31]:
# Query 1: GRAIN — one row really is one content item x one client x one day
grain_check_sql = f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
"""
con.sql(grain_check_sql)
# Expected: 0 rows back. Zero rows = the grain holds.

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │   c   │
│    date     │    varchar     │     varchar     │ int64 │
└─────────────┴────────────────┴─────────────────┴───────┘
                          0 rows                        

In [32]:

# Query 2: SLICE ROW COUNT + DATE SPAN
counts_sql = f"""
SELECT
  COUNT(*) AS row_count,
  COUNT(DISTINCT content_hash_id) AS distinct_content_items,
  COUNT(DISTINCT client_hash_id) AS distinct_clients,
  MIN(report_date) AS min_date,
  MAX(report_date) AS max_date
FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
"""
con.sql(counts_sql)

┌───────────┬────────────────────────┬──────────────────┬────────────┬────────────┐
│ row_count │ distinct_content_items │ distinct_clients │  min_date  │  max_date  │
│   int64   │         int64          │      int64       │    date    │    date    │
├───────────┼────────────────────────┼──────────────────┼────────────┼────────────┤
│   9841378 │                 331437 │               55 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────────────────┴──────────────────┴────────────┴────────────┘

In [33]:
# Query 3: AVAILABILITY — filter with IS TRUE (never = TRUE or NOT flag, because of NULLs)
availability_sql = f"""
SELECT
  COUNT(*) AS total_rows,
  COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
  COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
  COUNT(*) FILTER (WHERE gsc_data_available IS NULL) AS gsc_flag_null_rows
FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
"""
con.sql(availability_sql)

┌────────────┬────────────────────┬────────────────────┬────────────────────┐
│ total_rows │ gsc_available_rows │ ga4_available_rows │ gsc_flag_null_rows │
│   int64    │       int64        │       int64        │       int64        │
├────────────┼────────────────────┼────────────────────┼────────────────────┤
│    9841378 │            3611061 │             413966 │                  0 │
└────────────┴────────────────────┴────────────────────┴────────────────────┘

In [34]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 5 FEATURES — built only from the FEATURE WINDOW (2026-03-01 -> 2026-03-15), the decision point
# The LABEL WINDOW (2026-03-16 -> 2026-03-31) is kept separate on purpose, so nothing "future"
# leaks into a feature by accident.

feature_frame_sql = f"""
WITH fh AS (
  SELECT
    client_hash_id,
    content_hash_id,
    AVG(gsc_impressions) AS fh_avg_daily_impressions,
    AVG(gsc_avg_position) AS fh_avg_position,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END) AS fh_engaged_sessions,
    SUM(gsc_impressions) AS fh_total_impressions
  FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
  WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
    AND gsc_data_available IS TRUE
  GROUP BY client_hash_id, content_hash_id
),
sh AS (
  -- second half: used ONLY to build the label, kept apart from features
  SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS sh_total_impressions
  FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
  WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
    AND gsc_data_available IS TRUE
  GROUP BY client_hash_id, content_hash_id
),
content_meta AS (
  SELECT
    client_hash_id,
    content_hash_id,
    word_count,
    DATE_DIFF('day', content_created_date, DATE '2026-03-15') AS content_age_days
  FROM read_parquet('{BASE}/dim_content.parquet')
)
SELECT
  fh.client_hash_id,
  fh.content_hash_id,
  fh.fh_avg_daily_impressions,
  fh.fh_avg_position,
  fh.fh_engaged_sessions,
  cm.content_age_days,
  cm.word_count,
  fh.fh_total_impressions,
  COALESCE(sh.sh_total_impressions, 0) AS sh_total_impressions,
  CASE
    WHEN COALESCE(sh.sh_total_impressions, 0) < 0.8 * fh.fh_total_impressions THEN 1
    ELSE 0
  END AS is_declining
FROM fh
JOIN content_meta cm USING (client_hash_id, content_hash_id)
LEFT JOIN sh USING (client_hash_id, content_hash_id)
WHERE fh.fh_total_impressions > 0
"""

feature_df = con.sql(feature_frame_sql).df()
print(feature_df.shape)
feature_df.head(10)

(151981, 10)


,client_hash_id,content_hash_id,fh_avg_daily_impressions,fh_avg_position,fh_engaged_sessions,content_age_days,word_count,fh_total_impressions,sh_total_impressions,is_declining
0,client_23a62021009f63c4,content_af944cf4db332836,50.133333,6.287823,0.0,213,4942,752.0,520.0,1
1,client_23a62021009f63c4,content_3b363a01ccbd0d40,495.000000,27.873733,1.0,213,3616,7425.0,5962.0,0
2,client_23a62021009f63c4,content_1c8218d099fde1d8,2.571429,22.198299,0.0,213,5749,36.0,29.0,0
3,client_23a62021009f63c4,content_7b5a096402286b01,268.666667,27.948421,0.0,213,3214,4030.0,2740.0,1
4,client_23a62021009f63c4,content_2d5675beb712e0ce,249.933333,26.035111,2.0,213,3035,3749.0,2747.0,1
5,client_23a62021009f63c4,content_351c5f4fc8a27a65,211.733333,11.990970,3.0,213,3396,3176.0,4627.0,0
6,client_23a62021009f63c4,content_f241edd11baf55c9,23.466667,19.840580,0.0,213,4923,352.0,508.0,0
7,client_23a62021009f63c4,content_36cb55893b73e21c,500.733333,18.702115,6.0,213,3176,7511.0,8210.0,0
8,client_23a62021009f63c4,content_d57c484730f7ed62,549.333333,28.386054,0.0,213,3095,8240.0,4294.0,1
9,client_23a62021009f63c4,content_4f8bfa4de3664a60,14.600000,17.657933,0.0,213,4917,219.0,224.0,0


**The 5 features and why each is knowable at the decision moment (2026-03-15):**

1. `fh_avg_daily_impressions` — average daily GSC impressions, 03-01→03-15 only.
   Available because it's search performance already observed before the decision date.
2. `fh_avg_position` — average GSC ranking position over the same window.
   Available because ranking position is measured in the past, not predicted.
3. `fh_engaged_sessions` — total GA4 engaged sessions, 03-01→03-15 only.
   Available because it's actual user behavior that already happened.
4. `content_age_days` — days between `content_created_date` and 2026-03-15.
   Available because publish date is fixed the moment content goes live.
5. `word_count` — static content attribute from `dim_content`.
   Available because it's set when the page is authored, independent of any future window.

In [35]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_features = [
    "fh_avg_daily_impressions", "fh_avg_position",
    "fh_engaged_sessions", "content_age_days", "word_count"
]
y = feature_df["is_declining"]

# --- HONEST: only the 5 features knowable at the decision moment ---
X_honest = feature_df[honest_features].fillna(0)
Xh_train, Xh_test, y_train, y_test = train_test_split(
    X_honest, y, test_size=0.3, random_state=42, stratify=y
)
honest_model = LogisticRegression(max_iter=1000)
honest_model.fit(Xh_train, y_train)
honest_auc = roc_auc_score(y_test, honest_model.predict_proba(Xh_test)[:, 1])
print(f"HONEST AUC (5 features only): {honest_auc:.3f}")

# --- THE TRAP: add sh_total_impressions — it's literally what the label is built from ---
leaky_features = honest_features + ["sh_total_impressions"]
X_leaky = feature_df[leaky_features].fillna(0)
Xl_train, Xl_test, _, _ = train_test_split(
    X_leaky, y, test_size=0.3, random_state=42, stratify=y
)
leaky_model = LogisticRegression(max_iter=1000)
leaky_model.fit(Xl_train, y_train)
leaky_auc = roc_auc_score(y_test, leaky_model.predict_proba(Xl_test)[:, 1])
print(f"LEAKY AUC  (+ sh_total_impressions, a label-window column): {leaky_auc:.3f}")

print(f"\nJump: {honest_auc:.3f} -> {leaky_auc:.3f}")
print("sh_total_impressions is literally the number the label was thresholded on --")
print("of course a model 'discovers' it perfectly. This is the leakage lesson from ML-02,")
print("now reproduced on real warehouse data.")

# --- Delete the leak column and keep the honest number ---
feature_df = feature_df.drop(columns=["sh_total_impressions"])
print(f"\nFinal honest AUC kept for this notebook: {honest_auc:.3f}")

HONEST AUC (5 features only): 0.555
LEAKY AUC  (+ sh_total_impressions, a label-window column): 0.963

Jump: 0.555 -> 0.963
sh_total_impressions is literally the number the label was thresholded on --
of course a model 'discovers' it perfectly. This is the leakage lesson from ML-02,
now reproduced on real warehouse data.

Final honest AUC kept for this notebook: 0.555


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation:** This slice's clients have wildly uneven history depth (an unbalanced
panel — `dim_clients.gsc_data_start` differs per client). A content item that only has GSC
data starting mid-March will show artificially low `fh_avg_daily_impressions` — not because it's
performing worse, but because it wasn't tracked for the full first-half window. This notebook
doesn't correct for that; a future pass should filter to items with a full 15-day feature window,
or normalize by days-observed rather than raw sums/averages.

In [36]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Quick check: how many content items in our feature frame had FEWER than 15 days
# of GSC-available rows in the feature window (i.e. partial history, not a full window)?
days_observed_sql = f"""
SELECT
  client_hash_id,
  content_hash_id,
  COUNT(*) AS days_observed_in_first_half
FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
  AND gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
"""
days_df = con.sql(days_observed_sql).df()
partial = (days_df["days_observed_in_first_half"] < 15).mean()
print(f"Share of content items with a PARTIAL first-half window (<15 days): {partial:.1%}")

Share of content items with a PARTIAL first-half window (<15 days): 55.1%


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.